# Sentence-End Detection — Teacher-Student Feature Analysis

**Цель:** XGBoost teacher учится на акустике (Swift/vDSP) + MFA фонетике.
SHAP показывает вклад каждой фичи → student Swift-модель использует только те,
что вычислимы в рантайме.

**Dataset:** `sentence_end_teacher_features.csv`
- 14 acoustic features (Swift/vDSP, runtime-computable)
- 8 MFA phonetic features (teacher-only: phone_dur_ratio, n_phones, word_dur_ratio, ...)
- Labels: acoustically cleaned (pause≥50ms OR energy_slope<-0.001 OR rms_ratio>1.2)
- 75,762 rows, 10.5% positive rate

**Ключевой результат:**
- Teacher AUC (22 features): 0.791
- Student AUC (acoustic only): 0.718
- Gap: 0.073 → объясняется word_dur_ratio (MFA, SHAP 0.43) и n_phones (SHAP 0.31)

In [ ]:
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings, subprocess
warnings.filterwarnings('ignore')

gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'CPU only')
DEVICE = 'cuda' if gpu else 'cpu'

## 1. Загрузка данных — teacher (acoustic + MFA)

In [ ]:
import os, urllib.request

REPO = 'https://raw.githubusercontent.com/russianoracle/sentence-end-training/main'
CSV  = 'sentence_end_teacher_features.csv'

if not os.path.exists(CSV):
    print(f'Downloading {CSV}...')
    urllib.request.urlretrieve(f'{REPO}/{CSV}', CSV)

df = pd.read_csv(CSV)

ACOUSTIC = ['pause_sec','log_pause','rms_before','rms_after','rms_ratio',
            'energy_slope','rate_before','rate_after','rate_delta',
            'vowel_energy_ratio','spectral_flux','pitch_mean','pitch_drop','voiced_fraction']
MFA_REAL  = ['phone_dur_ratio','last_phone_dur','last_vowel_dur','vowel_ratio',
             'n_phones','phone_variability','word_dur','word_dur_ratio_mfa']
TEACHER_FEATURES = ACOUSTIC + MFA_REAL

print(f'Rows: {len(df):,}  |  pos: {df["y"].mean()*100:.1f}%')
print(f'Acoustic features ({len(ACOUSTIC)}): {ACOUSTIC}')
print(f'MFA features ({len(MFA_REAL)}):      {MFA_REAL}')
df.groupby('source')['y'].agg(['count','sum','mean']).rename(
    columns={'count':'total','sum':'pos_n','mean':'pos_rate'})

## 2. Teacher: XGBoost на всех 22 фичах (acoustic + MFA)

In [ ]:
X = df[TEACHER_FEATURES].fillna(0).values
y = df['y'].values
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
pos_w = (y_tr==0).sum()/(y_tr==1).sum()

teacher = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_w,
    tree_method='hist', device=DEVICE, random_state=42, verbosity=0
)
teacher.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
teacher_prob = teacher.predict_proba(X_te)[:,1]
auc_teacher  = roc_auc_score(y_te, teacher_prob)
print(f'Teacher AUC (22 features): {auc_teacher:.4f}')

## 3. SHAP — что важно и где MFA vs акустика

In [ ]:
explainer   = shap.TreeExplainer(teacher)
shap_values = explainer.shap_values(X_te)
mean_shap   = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'feature': TEACHER_FEATURES,
    'mean_shap': mean_shap,
    'type': ['acoustic']*len(ACOUSTIC) + ['MFA']*len(MFA_REAL)
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)

# Plot
colors = ['#2196F3' if t == 'acoustic' else '#FF5722' for t in shap_df['type']]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(shap_df['feature'], shap_df['mean_shap'], color=colors)
ax.set_xlabel('mean |SHAP|')
ax.set_title('Teacher SHAP — acoustic (blue) vs MFA phonetic (orange)')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2196F3',label='acoustic (runtime)'),
                   Patch(color='#FF5722',label='MFA phonetic (teacher-only)')])
plt.tight_layout(); plt.show()

print(shap_df.to_string(index=False))

## 4. Acoustic student vs Teacher

**Вывод из SHAP:**
- `word_dur_ratio_mfa` SHAP=0.43 и `n_phones` SHAP=0.31 — топ MFA фичи
- Обе требуют реализации в Swift: inter-boundary timing + rolling speaker mean
- Без них acoustic student потолок ≈ 0.718

In [ ]:
# Acoustic-only student (subset of features)
idx_ac = [TEACHER_FEATURES.index(f) for f in ACOUSTIC]

student = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_w,
    tree_method='hist', device=DEVICE, random_state=42, verbosity=0
)
student.fit(X_tr[:,idx_ac], y_tr, verbose=False)
student_prob = student.predict_proba(X_te[:,idx_ac])[:,1]
auc_student  = roc_auc_score(y_te, student_prob)

# LR student (для Swift export)
sc = StandardScaler()
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
lr.fit(sc.fit_transform(X_tr[:,idx_ac]), y_tr)
lr_prob = lr.predict_proba(sc.transform(X_te[:,idx_ac]))[:,1]
auc_lr  = roc_auc_score(y_te, lr_prob)

print(f'Teacher XGB  (22 features): {auc_teacher:.4f}')
print(f'Student XGB  (14 acoustic): {auc_student:.4f}')
print(f'Student LR   (14 acoustic): {auc_lr:.4f}')
print(f'Gap teacher→student:        {auc_teacher-auc_student:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
for name, prob in [('Teacher XGB (22)', teacher_prob),
                   ('Student XGB (14 acoustic)', student_prob),
                   ('Student LR (14 acoustic)', lr_prob)]:
    fpr, tpr, _ = roc_curve(y_te, prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_te,prob):.3f})')
ax.plot([0,1],[0,1],'k--',alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC: teacher vs student')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 5. Экспорт LR student weights для Swift

In [ ]:
import json

swift_model = {
    'coef':      lr.coef_[0].tolist(),
    'intercept': float(lr.intercept_[0]),
    'scaler': {
        'mean':  sc.mean_.tolist(),
        'scale': sc.scale_.tolist(),
    },
    'features': ACOUSTIC,
    'auc_roc':  float(auc_lr),
    'note': 'Student: acoustic-only. word_dur_ratio_mfa+n_phones pending Swift impl → +0.07 AUC',
}

with open('sentence_end_model_colab.json', 'w') as f:
    json.dump(swift_model, f, indent=2)

print('Saved: sentence_end_model_colab.json')
print(f'AUC:      {swift_model["auc_roc"]:.4f}')
print(f'Features: {ACOUSTIC}')

try:
    from google.colab import files
    files.download('sentence_end_model_colab.json')
except:
    pass